<a href="https://colab.research.google.com/github/Not-kh-lily-23/pulsar-conformal-triage/blob/main/baseline_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import os
import pandas as pd
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,roc_auc_score,average_precision_score,classification_report
from google.colab import drive
drive.mount('/content/drive')
base="pulsar-conformal-triage"
processed = '/content/drive/MyDrive/pulsar_project/data/processed'
print("Loading the processed splits...")
try:
    trn_df=pd.read_csv(f"{processed}/train.csv")
    tst_df=pd.read_csv(f"{processed}/test.csv")
except FileNotFoundError:
    raise Exception("Data not found")
x_train=trn_df.drop(columns=["target"])
y_train=trn_df["target"]
x_test=tst_df.drop(columns=["target"])
y_test=tst_df["target"]
print(f"Train set: {x_train.shape[0]} rows | Test set: {x_test.shape[0]} rows\n")
mdl={
    "LightGBM": lgb.LGBMClassifier(random_state=42,n_estimators=100,verbose=-1),
    "Logistic Regression": LogisticRegression(random_state=42,max_iter=1000)
}
for name,mdl in mdl.items():
    print(f"Training {name}")
    mdl.fit(x_train,y_train)
    y_probs=mdl.predict_proba(x_test)[:,1]
    y_preds=(y_probs>=0.5).astype(int)
    acc=accuracy_score(y_test,y_preds)
    roc_auc=roc_auc_score(y_test,y_probs)
    pr_auc=average_precision_score(y_test,y_probs)
    print(f"Accuracy: {acc:.4f}")
    print(f"ROC-AUC:  {roc_auc:.4f}")
    print(f"PR-AUC:   {pr_auc:.4f}")
    print("\nClassification Report (Look at the recall for Class 1!):")
    print(classification_report(y_test,y_preds))

Mounted at /content/drive
Loading the processed splits...
Train set: 10738 rows | Test set: 3580 rows

Training LightGBM
Accuracy: 0.9799
ROC-AUC:  0.9753
PR-AUC:   0.9263

Classification Report (Look at the recall for Class 1!):
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3252
           1       0.92      0.86      0.89       328

    accuracy                           0.98      3580
   macro avg       0.95      0.92      0.94      3580
weighted avg       0.98      0.98      0.98      3580

Training Logistic Regression
Accuracy: 0.9793
ROC-AUC:  0.9719
PR-AUC:   0.9335

Classification Report (Look at the recall for Class 1!):
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3252
           1       0.94      0.82      0.88       328

    accuracy                           0.98      3580
   macro avg       0.96      0.91      0.93      3580
weighted avg       0.98      0.98